<a href="https://colab.research.google.com/github/Somaskandan931/flyrank-ml-2026-Somaskandan931/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Somaskandan931/flyrank-ml-2026-Somaskandan931/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding A — "What Predicts Health?" (Random Forest feature importance for Health Score, ML appendix p.27).**
Where the label comes from: Health Score isn't measured independently — the paper's own "Understanding the Metrics" page defines it as a formula: Impressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll Depth (20 pts). My question: the reported top three importances — Position 43%, Impressions 32%, Scroll Depth 15% — sum to 90%, and those are three of the label's own four ingredients. The paper is careful to note that "importance is descriptive rather than causal," which I'd credit, but a holdout split doesn't rescue this on its own — holdout tests whether a fit generalizes, not whether the fit found anything beyond the label's arithmetic. Per the leakage taxonomy's label-derived-feature test, the confirming move would be reporting accuracy WITH vs WITHOUT those four formula terms (predicting Health Score from Clicks, Age, Word Count, Days Visible, AI Sessions alone) — if that collapses toward the naive mean, it shows the model isn't finding anything the formula didn't already hand it. I'd suggest that as a companion chart, not a rewrite of this one.

**Finding B — "What Predicts Growth?" (logistic regression, 71% holdout accuracy, p.28-29).**
Where the label comes from: the growing/declining label comes from Trend Direction, itself computed from the 30-day-vs-previous-30-day impression change ("Up" = >10% growth). Three methodology questions, in the same spirit as the leakage skill's checklist:
1. *Base rate missing.* "71% holdout accuracy" is reported with no growing/declining split anywhere nearby. Per the skill's base-rate rule, 71% on a 60/40 label is a very different result from 71% on a 50/50 label, and a reader can't tell which this is.
2. *Possible window overlap.* The prose singles out "recent impressions" as one of the strongest positive predictors of growth, but the label is itself defined from a 30-day impression trend. If "recent impressions" is measured over a window overlapping the trend-calculation window, the model may be partly reading its own label rather than finding an independent signal — the paper doesn't disclose the exact window, so this can't be ruled out.
3. *Split type undisclosed.* With 57 brands in the portfolio, a plain random holdout would let pages from the same brand sit on both sides of the split, letting the model partly memorize brand-level character instead of genuinely generalizing. The paper doesn't say whether the 71% comes from a random or brand-grouped split.

None of this is a "gotcha" — this paper already discloses more of its own limits than most public reports do (it separates "exploratory appendix" from "headline findings" and flags the Health-Score circularity itself). These are the same three questions I turn on my own model next.

In [ ]:
import pandas as pd

# Numbers copied directly from the paper's own charts/pages (pp. 5, 27-29) --
# used here to CHECK the two findings above, not to re-derive them.

health_score_formula_weights = {  # "Understanding the Metrics" page, weights out of 100
    "Position": 30, "Impressions": 30, "CTR": 20, "Scroll Depth": 20,
}
health_rf_importance = {  # Random Forest feature importance for Health Score, p.27
    "Average Position": 43, "Impressions": 32, "Scroll Depth": 15, "CTR": 8,
    "Clicks": 2, "Sessions": 0, "Content Age": 0, "Word Count": 0,
    "Days Visible": 0, "AI Sessions": 0,
}

formula_terms = {"Position", "Impressions", "CTR", "Scroll Depth"}
share_from_formula = sum(
    v for k, v in health_rf_importance.items()
    if k.replace("Average ", "") in formula_terms
)
print(f"Finding A check -- share of RF importance coming from the label's own "
      f"formula terms: {share_from_formula}% of 100")
print("  -> methodology question: is there a version of this chart with those terms removed?\n")

# Finding B: what's disclosed vs. what a reader would need to trust "71% holdout accuracy"
growth_audit = pd.DataFrame({
    "check": [
        "Growing/declining base rate disclosed?",
        "Holdout split type disclosed (random vs brand-grouped)?",
        "'Recent impressions' window vs. label's 30d trend window disclosed?",
    ],
    "found_in_paper": ["No", "No", "No"],
    "why_it_matters": [
        "71% accuracy is unreadable without the majority-class rate next to it",
        "57 brands + a random split risks brand-level memorization, not generalization",
        "an overlapping window would mean the model partly reads its own label",
    ],
})
growth_audit

Finding A check -- share of RF importance coming from the label's own formula terms: 98% of 100
  -> methodology question: is there a version of this chart with those terms removed?



,check,found_in_paper,why_it_matters
0,Growing/declining base rate disclosed?,No,71% accuracy is unreadable without the majorit...
1,Holdout split type disclosed (random vs brand-...,No,57 brands + a random split risks brand-level m...
2,'Recent impressions' window vs. label's 30d tr...,No,an overlapping window would mean the model par...


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

My Week-5 model (`w05_model.ipynb`) already trains on a client-grouped split (`GroupShuffleSplit` on `client_hash_id`), because a plain random split would let rows from the same client leak systemic character across train/test. To make that choice visible rather than assumed, this section reruns the exact same data, features, and label under **both** a naive random split (no grouping — what a less careful version of ML-08 could have shipped) and the honest grouped split, side by side, on the same model (Random Forest). The gap between the two rows is itself evidence of how much a random split would have let this model quietly memorize, per the leakage skill's guidance to report both numbers rather than only the honest one.

In [ ]:
%pip install -q duckdb huggingface_hub pandas scikit-learn

from huggingface_hub import login
from google.colab import userdata
import duckdb, pandas as pd, numpy as np, os

login(token=userdata.get('HF_TOKEN'))
con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql("CREATE SECRET (TYPE huggingface, TOKEN '" + userdata.get('HF_TOKEN') + "')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
FACT_DAILY = f"{BASE}/fact_content_daily_performance/**/*.parquet"

# Same config as w05_model.ipynb -- same data, same label, same features.
GROUP_COL = "client_hash_id"
LABEL_COL = "label_high_clicks_late"
FEATURE_COLS = [
    "gsc_impressions_early", "gsc_clicks_early",
    "gsc_avg_position_early", "ga4_sessions_early", "scroll_events_early",
]
K_FOR_PRECISION = 50
RANDOM_STATE = 42
EARLY_START, EARLY_END = "2026-03-01", "2026-03-15"
LATE_START, LATE_END = "2026-03-16", "2026-03-31"

df = con.sql(f"""
WITH early AS (
    SELECT content_hash_id, client_hash_id,
           SUM(gsc_impressions) AS gsc_impressions_early,
           SUM(gsc_clicks)      AS gsc_clicks_early,
           SUM(gsc_sum_position) / NULLIF(SUM(gsc_impressions), 0) AS gsc_avg_position_early,
           SUM(ga4_sessions)    AS ga4_sessions_early,
           SUM(scroll_events)   AS scroll_events_early
    FROM read_parquet('{FACT_DAILY}')
    WHERE month = '2026-03'
      AND report_date BETWEEN '{EARLY_START}' AND '{EARLY_END}'
      AND gsc_data_available IS TRUE
    GROUP BY 1,2
),
late AS (
    SELECT content_hash_id, client_hash_id,
           SUM(gsc_clicks) AS gsc_clicks_late
    FROM read_parquet('{FACT_DAILY}')
    WHERE month = '2026-03'
      AND report_date BETWEEN '{LATE_START}' AND '{LATE_END}'
      AND gsc_data_available IS TRUE
    GROUP BY 1,2
)
SELECT e.*, l.gsc_clicks_late
FROM early e
JOIN late l USING (content_hash_id, client_hash_id)
""").df()

df = df.dropna(subset=FEATURE_COLS + ["gsc_clicks_late"])
median_clicks = df["gsc_clicks_late"].median()
df[LABEL_COL] = (df["gsc_clicks_late"] > median_clicks).astype(int)
print(f"Loaded {df.shape[0]} rows, {df[GROUP_COL].nunique()} clients, base rate {df[LABEL_COL].mean():.3f}")

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

def precision_at_k(y_true, scores, k):
    order = np.argsort(scores)[::-1][:k]
    return np.asarray(y_true)[order].mean()

def run_split(name, train_idx, test_idx):
    train_df, test_df = df.iloc[train_idx], df.iloc[test_idx]
    overlap = set(train_df[GROUP_COL]) & set(test_df[GROUP_COL])
    model = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=RANDOM_STATE,
                                    class_weight="balanced", n_jobs=-1)
    model.fit(train_df[FEATURE_COLS], train_df[LABEL_COL])
    scores = model.predict_proba(test_df[FEATURE_COLS])[:, 1]
    return {
        "split": name,
        "clients_shared_train_test": len(overlap),
        f"precision_at_{K_FOR_PRECISION}": round(precision_at_k(test_df[LABEL_COL], scores, K_FOR_PRECISION), 3),
        "roc_auc": round(roc_auc_score(test_df[LABEL_COL], scores), 3),
        "avg_precision": round(average_precision_score(test_df[LABEL_COL], scores), 3),
    }

# BEFORE: naive random split, no grouping -- a less careful version of ML-08 could ship this
naive_train_idx, naive_test_idx = train_test_split(
    np.arange(len(df)), test_size=0.2, random_state=RANDOM_STATE, stratify=df[LABEL_COL]
)

# AFTER: the honest split my w05 model actually used -- grouped by client
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
honest_train_idx, honest_test_idx = next(splitter.split(df, groups=df[GROUP_COL]))

comparison = pd.DataFrame([
    run_split("before_naive_random", naive_train_idx, naive_test_idx),
    run_split("after_honest_grouped", honest_train_idx, honest_test_idx),
]).set_index("split")
comparison

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Loaded 75222 rows, 33 clients, base rate 0.372


,clients_shared_train_test,precision_at_50,roc_auc,avg_precision
split,,,,
before_naive_random,32,1.0,0.877,0.846
after_honest_grouped,0,1.0,0.882,0.832


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Same checklist as `w03_feature_leakage_check.ipynb` (ML-05), now run against the five columns that actually shipped in the Week-5 model: `gsc_impressions_early`, `gsc_clicks_early`, `gsc_avg_position_early`, `ga4_sessions_early`, `scroll_events_early`, against label `label_high_clicks_late`.

- **Timeline:** features come only from March 1-15 (`EARLY_*`), label only from March 16-31 (`LATE_*`) — no shared dates, asserted below.
- **Label-derived/sibling columns:** none of the five features is a rename or sub-aggregate of `gsc_clicks_late` — the closest is `gsc_clicks_early`, but it's a disjoint time window, not the same column.
- **Product flags:** none of FlyRank's own workflow labels (Healthy / Fix CTR / Fix Content / Zombie Page) or the Health Score composite from Section 1 are anywhere in `FEATURE_COLS`.
- **Population selection:** the `gsc_data_available IS TRUE` filter is applied independently inside the `early` and `late` CTEs, using only the flag for that CTE's own window — it never reaches into the other window's outcome to decide who's in the population.
- **Group column:** `client_hash_id` never appears in `FEATURE_COLS`, only used for splitting.

The one thing the checklist can't confirm by inspection alone is whether any of the five still gives the model an unrealistic edge — that needs the train-with/train-without test below. If any `auc_drop_vs_full` comes back large and its own presence alone gives a near-1.0 score, that's the confession per the leakage skill and deserves a closer look before the feature ships again.

In [ ]:
# 1. Timeline check
assert EARLY_END < LATE_START, "Feature window overlaps label window"
print(f"Feature window: {EARLY_START} to {EARLY_END} | Label window: {LATE_START} to {LATE_END} -> no overlap")

# 2. No ID/group columns snuck into features
assert GROUP_COL not in FEATURE_COLS
assert "content_hash_id" not in FEATURE_COLS
print("Group/ID columns absent from FEATURE_COLS: OK")

# 3. Base rate, printed next to every metric from here on
print(f"Base rate (label positive): {df[LABEL_COL].mean():.3f}")

# 4. Train-once-WITH, train-once-WITHOUT each feature -- the label-derived-feature test,
#    run on the same honest (client-grouped) split from Section 2.
train_df2, test_df2 = df.iloc[honest_train_idx], df.iloc[honest_test_idx]

rows = []
base_model = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=RANDOM_STATE,
                                     class_weight="balanced", n_jobs=-1)
base_model.fit(train_df2[FEATURE_COLS], train_df2[LABEL_COL])
full_auc = roc_auc_score(test_df2[LABEL_COL], base_model.predict_proba(test_df2[FEATURE_COLS])[:, 1])
rows.append({"features": "all 5 (full)", "roc_auc": round(full_auc, 3), "auc_drop_vs_full": 0.0})

for suspect in FEATURE_COLS:
    remaining = [c for c in FEATURE_COLS if c != suspect]
    m = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=RANDOM_STATE,
                                class_weight="balanced", n_jobs=-1)
    m.fit(train_df2[remaining], train_df2[LABEL_COL])
    auc_without = roc_auc_score(test_df2[LABEL_COL], m.predict_proba(test_df2[remaining])[:, 1])
    rows.append({
        "features": f"without {suspect}",
        "roc_auc": round(auc_without, 3),
        "auc_drop_vs_full": round(full_auc - auc_without, 3),
    })

leak_check = pd.DataFrame(rows)
leak_check

Feature window: 2026-03-01 to 2026-03-15 | Label window: 2026-03-16 to 2026-03-31 -> no overlap
Group/ID columns absent from FEATURE_COLS: OK
Base rate (label positive): 0.372


,features,roc_auc,auc_drop_vs_full
0,all 5 (full),0.882,0.000
1,without gsc_impressions_early,0.843,0.039
2,without gsc_clicks_early,0.870,0.012
3,without gsc_avg_position_early,0.874,0.008
4,without ga4_sessions_early,0.881,0.000
5,without scroll_events_early,0.883,-0.001


## 4. Claim rewrite

*Take my own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

My boldest sentence so far lives in `w05_model.ipynb`'s Section 3 note — it describes the random forest as delivering a "lift" over the baseline, phrasing that reads like a causal guarantee rather than a measured, out-of-sample comparison. Below I pull the actual honest-split numbers from Section 2 and rewrite that sentence at the level the claim ladder actually supports (a validated model ranking out-of-sample, not a causal claim about future search performance), then run it through a banned-phrase check.

In [ ]:
honest_row = comparison.loc["after_honest_grouped"]
precision_col = f"precision_at_{K_FOR_PRECISION}"

bold_claim = (
    f"The model lifts Precision@{K_FOR_PRECISION} to {honest_row[precision_col]}, "
    f"proving it predicts which pages will grow clicks."
)

safe_claim = (
    f"On a client-held-out split, the Random Forest ranks the top {K_FOR_PRECISION} scored pages "
    f"with a measured Precision@{K_FOR_PRECISION} of {honest_row[precision_col]} and ROC AUC of "
    f"{honest_row['roc_auc']} (base rate {df[LABEL_COL].mean():.2f}). That is an observed, "
    f"decision-support signal for which early-window pages look worth reviewing first -- "
    f"not a causal claim about future click growth."
)

banned_phrases = ["proves", "causes", "will increase", "predicts which", "guarantees"]

print("BOLD (as first drafted):\n", bold_claim, "\n")
print("REWRITTEN (claim-ladder safe):\n", safe_claim, "\n")

hits = [p for p in banned_phrases if p in safe_claim.lower()]
assert not hits, f"Rewritten claim still contains banned language: {hits}"
print("Banned-phrase self-check: PASS -- rewritten claim is clean")

BOLD (as first drafted):
 The model lifts Precision@50 to 1.0, proving it predicts which pages will grow clicks. 

REWRITTEN (claim-ladder safe):
 On a client-held-out split, the Random Forest ranks the top 50 scored pages with a measured Precision@50 of 1.0 and ROC AUC of 0.882 (base rate 0.37). That is an observed, decision-support signal for which early-window pages look worth reviewing first -- not a causal claim about future click growth. 

Banned-phrase self-check: PASS -- rewritten claim is clean


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.